# 03 - Modules, Packages, I/O & the Standard Library

**Sources:** Bootcamp BP2 "Packages & Modules" (import mechanics, `__init__.py`, namespace
packages) and Summer School *CursP_BasePython* (scripts, `sys.argv`, `__main__`, modules, file I/O,
`os`, `os.path`, `glob`, `sys`).

**Contents:** what a module is -> the import machinery -> packages -> scripts vs modules ->
files with `pathlib` -> `os` / `sys` / `glob` -> a tour of useful stdlib modules.

## 1. Modules
A **module** is a `.py` file; its name is the filename without `.py`. Importing it *executes* the file
once and gives you a **namespace object**.

In [1]:
import math
import os
import sys

print(type(math), math.sqrt(16), math.pi)
print(math.__name__, "| functions in math:", len([n for n in dir(math) if not n.startswith("_")]))

# Ways to import
from math import sqrt, floor            # import specific names
import numpy as np                      # alias (a convention: np, pd, plt)
# from math import *                    # AVOID: star-imports pollute the namespace and hide where names come from

<class 'module'> 4.0 3.141592653589793
math | functions in math: 61


## 2. What happens on `import foo`? (BP2)

1. **Search** `sys.modules` (cache). If present, reuse it.
2. Else search the directories in `sys.path` (script dir / cwd, `PYTHONPATH`, stdlib, site-packages).
3. **Compile** to bytecode (cached in `__pycache__/foo.cpython-312.pyc`) if necessary.
4. **Execute** the module code in a new namespace (defines functions, classes, variables).
5. **Store** it in `sys.modules`, then **bind** the name in the importing namespace.

Consequence: **module code runs only once per process** - later imports are cache hits.

In [2]:
print(sys.path[:4], "...")
print("math" in sys.modules, "json" in sys.modules)
import json
print("json" in sys.modules)

['C:\\Users\\leire\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\leire\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\leire\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'C:\\Users\\leire\\AppData\\Local\\Programs\\Python\\Python312'] ...
True True
True


Let's *prove* the "evaluated only once" rule by creating a module on the fly in a temporary folder.

In [3]:
import importlib
import tempfile
import textwrap
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
workdir = Path(tmp.name)
(workdir / "demo.py").write_text(textwrap.dedent('''
    """A demo module."""
    print(">>> demo.py is being executed")

    c = 2

    def print_a():
        "Prints a."
        print("a")

    def print_b():
        "Prints b."
        print("b")
'''))
sys.path.insert(0, str(workdir))       # make the folder importable

import demo                            # prints the message (first import)
import demo                            # silent: served from sys.modules
demo.print_a(); demo.print_b()
print(demo.__doc__, demo.c, demo.__file__.endswith("demo.py"))

>>> demo.py is being executed
a
b
A demo module. 2 True


In [4]:
# Module caching warning: editing the file does NOT affect an already-imported module...
(workdir / "demo.py").write_text('print(">>> demo v2 executed")\nc = 3\n')
import demo
print("still old value:", demo.c)

# ...unless you reload explicitly (or restart the kernel)
importlib.reload(demo)
print("new value:", demo.c)

still old value: 2
>>> demo v2 executed
new value: 3


In [5]:
# Introspection
print([n for n in dir(demo) if not n.startswith("__")])
print(vars(demo).keys() >= {"c"})

['c', 'print_a', 'print_b']
True


## 3. Scripts vs modules: `if __name__ == "__main__"`
When a file is *run*, `__name__ == "__main__"`; when it is *imported*, `__name__` is the module name.
The guard lets the same file be both a reusable module and a runnable script.

In [6]:
(workdir / "demo2.py").write_text(textwrap.dedent('''
    import sys

    def print_a():
        "Prints a."
        print("a")

    def main(argv):
        print("args:", argv)
        print_a()

    if __name__ == "__main__":
        main(sys.argv[1:])
'''))
import demo2                              # nothing printed: only defines things
print("imported as:", demo2.__name__)

import subprocess
result = subprocess.run([sys.executable, str(workdir / "demo2.py"), "test", "arguments"],
                        capture_output=True, text=True)
print(result.stdout)

imported as: demo2
args: ['test', 'arguments']
a



## 4. Packages
A **package** is a directory of modules, normally with an `__init__.py`. Dotted names mirror the folders.

```text
sound/                  top-level package
    __init__.py         runs on `import sound`; defines the public API
    formats/            sub-package
        __init__.py
        wavread.py
        wavwrite.py
    effects/
        __init__.py
        echo.py
        surround.py
```

`__init__.py` purposes: (1) mark the folder as a package, (2) curate the public namespace
(`from .echo import add_echo`, `__all__ = [...]`), (3) run initialisation code on import.

**Namespace packages (PEP 420):** a folder *without* `__init__.py` can still be imported; several
folders on `sys.path` with the same name merge into one package - useful for plugin systems or
splitting a large distribution.

**Best practices:** small focused modules, descriptive names, document public APIs, avoid
*circular imports* (extract shared code to a third module or import inside the function),
prefer *absolute* imports (`from sound.effects import echo`) and explicit relative ones inside a
package (`from . import echo`).

In [7]:
# Build a tiny package and import from it
pkg = workdir / "shapes"
(pkg / "round").mkdir(parents=True)
(pkg / "__init__.py").write_text("from .basic import area_square\n__all__ = ['area_square']\n")
(pkg / "basic.py").write_text("def area_square(side):\n    return side * side\n")
(pkg / "round" / "__init__.py").write_text("")
(pkg / "round" / "circle.py").write_text(textwrap.dedent('''
    import math
    from ..basic import area_square          # explicit relative import

    def area(r):
        return math.pi * r * r

    def bounding_square_area(r):
        return area_square(2 * r)
'''))

import shapes
from shapes.round import circle
print(shapes.area_square(3), round(circle.area(1), 3), circle.bounding_square_area(1))

9 3.142 4


In [8]:
tmp.cleanup()
sys.path.remove(str(workdir))
for m in ["demo", "demo2", "shapes", "shapes.round", "shapes.round.circle", "shapes.basic"]:
    sys.modules.pop(m, None)

## 5. Files: always use `with`
`with open(...)` guarantees the file is closed even if an exception happens (the `try/finally` pattern
from notebook 02, packaged as a *context manager*).

In [9]:
import tempfile
from pathlib import Path

scratch = Path(tempfile.mkdtemp())
path = scratch / "workfile.txt"

with open(path, "w", encoding="utf-8") as f:       # 'w' truncate, 'a' append, 'x' create-only, 'r' read
    f.write("This is a test\nand another test\n")

with open(path, encoding="utf-8") as f:
    print(repr(f.read()))

with open(path, encoding="utf-8") as f:
    for line in f:                                 # lazy: one line at a time, even for huge files
        print(line.rstrip("\n"))

print(path.read_text(encoding="utf-8").splitlines())

'This is a test\nand another test\n'
This is a test
and another test
['This is a test', 'and another test']


### `pathlib` - paths as objects (prefer it over string-based `os.path`)

In [10]:
p = scratch / "data" / "run_01" / "result.csv"
print(p.name, p.stem, p.suffix, p.parent.name, p.parts[-3:])
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("x,y\n1,2\n")
print(p.exists(), p.is_file(), p.parent.is_dir(), p.stat().st_size, "bytes")
print(sorted(x.name for x in scratch.rglob("*")))
print(p.with_suffix(".json").name, Path.home().name != "", Path.cwd().name)

result.csv result .csv run_01 ('data', 'run_01', 'result.csv')
True True True 10 bytes
['data', 'result.csv', 'run_01', 'workfile.txt']
result.json True notebooks


### The same things with `os` / `os.path` (you will meet them in older code)

In [11]:
print(os.getcwd() == str(Path.cwd()))
print(os.listdir(scratch))
os.mkdir(scratch / "junkdir")
os.rename(scratch / "junkdir", scratch / "foodir")
print("foodir" in os.listdir(scratch))
os.rmdir(scratch / "foodir")

fp = scratch / "junk.txt"
fp.touch()
a = os.path.abspath(fp)
print(os.path.split(a)[1], os.path.dirname(a) == str(scratch), os.path.splitext(os.path.basename(a)))
print(os.path.exists(fp), os.path.isfile(fp), os.path.isdir(fp))
os.remove(fp)
print(os.path.join(os.path.expanduser("~"), "local", "bin"))

# Walk a directory tree
for dirpath, dirnames, filenames in os.walk(scratch):
    print(os.path.relpath(dirpath, scratch), filenames)

True
['data', 'workfile.txt']
True
junk.txt True ('junk', '.txt')
True True False
C:\Users\leire\local\bin
. ['workfile.txt']
data []
data\run_01 ['result.csv']


In [12]:
import glob
print([os.path.basename(x) for x in glob.glob(str(scratch / "*.txt"))])

['workfile.txt']


## 6. `sys`, command-line arguments, `argparse`

In [13]:
print(sys.platform, sys.version_info[:3], sys.maxsize)
print("argv in a notebook:", sys.argv[:1], "... (in a script it is [script, arg1, arg2, ...])")
sys.stdout.write("write without newline conversion\n")
sys.stderr.write("")   # errors should go to stderr

win32 (3, 12, 3) 9223372036854775807
argv in a notebook: ['C:\\Users\\leire\\Projects\\Python-Refresher\\.venv\\Lib\\site-packages\\ipykernel_launcher.py'] ... (in a script it is [script, arg1, arg2, ...])
write without newline conversion


0

In [14]:
import argparse

parser = argparse.ArgumentParser(description="Sum numbers")
parser.add_argument("numbers", nargs="+", type=float, help="numbers to add")
parser.add_argument("--round", type=int, default=2, help="decimals")
parser.add_argument("-v", "--verbose", action="store_true")
args = parser.parse_args(["1.5", "2.25", "--round", "1", "-v"])     # in a script: parse_args()
print(args, round(sum(args.numbers), args.round))

Namespace(numbers=[1.5, 2.25], round=1, verbose=True) 3.8


In [15]:
import shutil
shutil.rmtree(scratch, ignore_errors=True)     # clean up the scratch folder used above

## 7. Text formats: CSV and JSON

In [16]:
import csv
import io
import json

csv_text = "name;score\nana;9.5\nbob;7\n"
rows = list(csv.DictReader(io.StringIO(csv_text), delimiter=";"))
print(rows)

out = io.StringIO()
writer = csv.writer(out, delimiter=";")
writer.writerow(["name", "score"]); writer.writerow(["cy", 8])
print(out.getvalue())

record = {"name": "ana", "scores": [9.5, 7], "active": True, "nickname": None}
text = json.dumps(record, indent=2)
print(text)
print(json.loads(text) == record)

[{'name': 'ana', 'score': '9.5'}, {'name': 'bob', 'score': '7'}]
name;score
cy;8

{
  "name": "ana",
  "scores": [
    9.5,
    7
  ],
  "active": true,
  "nickname": null
}
True


## 8. A tour of the standard library ("batteries included")

In [17]:
# collections: specialised containers
from collections import Counter, defaultdict, deque, namedtuple, OrderedDict, ChainMap

Point = namedtuple("Point", "x y")
pt = Point(1, 2)
print(pt, pt.x, pt._replace(y=5), pt._asdict())

dq = deque([1, 2, 3], maxlen=3)        # O(1) at both ends (a queue / sliding window)
dq.append(4); dq.appendleft(0)
print(dq)
print(Counter("abracadabra").most_common(3))

Point(x=1, y=2) 1 Point(x=1, y=5) {'x': 1, 'y': 2}
deque([0, 2, 3], maxlen=3)
[('a', 5), ('b', 2), ('r', 2)]


In [18]:
# itertools: building blocks for iteration
import itertools as it

print(list(it.islice(it.count(10, 5), 4)))
print(list(it.chain([1, 2], "ab")))
print(list(it.permutations("abc", 2)))
print(list(it.combinations("abcd", 2)))
print(list(it.product([0, 1], repeat=3))[:4], "...")
print([(k, list(g)) for k, g in it.groupby("aabccc")])
print(list(it.accumulate([1, 2, 3, 4])))          # prefix sums! (Codility lesson 5)

[10, 15, 20, 25]
[1, 2, 'a', 'b']
[('a', 'b'), ('a', 'c'), ('b', 'a'), ('b', 'c'), ('c', 'a'), ('c', 'b')]
[('a', 'b'), ('a', 'c'), ('a', 'd'), ('b', 'c'), ('b', 'd'), ('c', 'd')]
[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1)] ...
[('a', ['a', 'a']), ('b', ['b']), ('c', ['c', 'c', 'c'])]
[1, 3, 6, 10]


In [19]:
# datetime, random, statistics, math
from datetime import datetime, timedelta, date
import random
import statistics

d0 = datetime(2026, 10, 9, 18, 45)
print(d0.strftime("%A %d %B %Y, %H:%M"), d0 + timedelta(days=30, hours=2), (date(2026, 12, 25) - date(2026, 10, 9)).days)

rng = random.Random(42)                # own generator -> reproducible
sample = [rng.randint(1, 100) for _ in range(10)]
print(sample, statistics.mean(sample), statistics.median(sample), round(statistics.stdev(sample), 2))
print(rng.sample(range(100), 3), rng.choice("abc"))

import math
print(math.gcd(24, 9), math.lcm(4, 6), math.comb(5, 2), math.factorial(5), math.isqrt(99), math.prod([1, 2, 3, 4]))

Friday 09 October 2026, 18:45 2026-11-08 20:45:00 77
[82, 15, 4, 95, 36, 32, 29, 18, 95, 14] 42 30.5 35.05
[86, 94, 69] a
3 12 10 120 9 24


In [20]:
# re (regular expressions), string, textwrap
import re

text = "Order #123 shipped on 2026-10-09; order #456 on 2026-10-11."
print(re.findall(r"#(\d+)", text))
print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", text))
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})", text)
print(m.groupdict())

['123', '456']
Order #123 shipped on 09/10/2026; order #456 on 11/10/2026.
{'year': '2026', 'month': '10'}


In [21]:
# logging instead of print for real programs
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s", force=True)
log = logging.getLogger("refresher")
log.debug("hidden (below INFO)")
log.info("visible %s", "message")
log.warning("something odd")

INFO:refresher:visible message


## Exercises (from the Summer School slides)

**A. Column statistics (10 min).** Write a script `stats_col.py` that reads a text file containing one
number per line and prints the min, max and sum. Handle blank lines and invalid numbers gracefully
(log a warning, continue). Accept the filename through `argparse`.

**B. Taylor series module (10 min).** Create a module `taylor.py` with `sin(x, terms=10)`, `cos(x, terms=10)`
and `exp(x, terms=20)` computed from their Taylor expansions, plus a `__main__` block that prints
the error against `math.sin` etc. for a few values.

**C. Directory report.** Using `pathlib`, print the 5 largest files below the current folder
with their sizes, and count files per extension with `Counter`.

In [22]:
# Your turn